# Build and Train a GPT from Scratch (nanoGPT)

This notebook trains a **character-level GPT** on Shakespeare from scratch.
Based on Andrej Karpathy's [nanoGPT](https://github.com/karpathy/nanoGPT).

**What we build:**
- A full GPT-2-style decoder-only transformer (`../nanogpt/model.py`)
- A character-level tokenizer (simplest possible)
- A training loop with gradient clipping and learning rate scheduling
- Text generation via temperature sampling and top-k

**Architecture overview:**
```
Input tokens (chars)
    ↓
Token Embedding + Positional Embedding
    ↓
[Block × N]:  LayerNorm → CausalSelfAttention → LayerNorm → MLP
    ↓          (each with residual connection)
Final LayerNorm
    ↓
Linear → Vocab logits → softmax → next token probabilities
```

**Training target:** At each position, predict the *next* character. Loss = cross-entropy.

In [ ]:
# !pip install torch requests

In [ ]:
import sys
sys.path.insert(0, '..')  # so we can import nanogpt/

import torch
import torch.nn as nn
import numpy as np
import requests
import os
import time
import math

from nanogpt import GPT, GPTConfig

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## Step 1: Dataset — Shakespeare

We use the classic `tiny shakespeare` dataset: ~1MB of all Shakespeare's works concatenated.

In [ ]:
DATA_PATH = '../data/shakespeare.txt'

if not os.path.exists(DATA_PATH):
    print("Downloading Shakespeare dataset...")
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    text = requests.get(url).text
    with open(DATA_PATH, 'w') as f:
        f.write(text)
    print(f"Downloaded {len(text):,} characters")
else:
    with open(DATA_PATH, 'r') as f:
        text = f.read()
    print(f"Loaded {len(text):,} characters")

print(f"\nFirst 300 characters:")
print(text[:300])

## Step 2: Character-level tokenizer

The simplest possible tokenizer: each unique character is a token.
No subword, no BPE — just map char → integer.

In [ ]:
# Build vocabulary
chars = sorted(set(text))
vocab_size = len(chars)
print(f"Vocabulary size: {vocab_size} characters")
print(f"Characters: {''.join(chars)!r}")

# Encoder/decoder
stoi = {c: i for i, c in enumerate(chars)}  # char → int
itos = {i: c for i, c in enumerate(chars)}  # int → char

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)

# Test roundtrip
test = "Hello, World!"
print(f"\nEncode: {test!r} → {encode(test)}")
print(f"Decode: {encode(test)} → {decode(encode(test))!r}")

In [ ]:
# Encode the full dataset
data = torch.tensor(encode(text), dtype=torch.long)
print(f"Dataset size: {len(data):,} tokens")

# Train/val split (90/10)
n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]
print(f"Train: {len(train_data):,} tokens")
print(f"Val:   {len(val_data):,} tokens")

## Step 3: Data loader

We sample random **chunks** of length `block_size` from the dataset.
The target (`y`) is the same chunk shifted one character to the right.

In [ ]:
def get_batch(split: str, block_size: int = 256, batch_size: int = 32):
    """Sample a random batch of (inputs, targets) from train or val data."""
    data_split = train_data if split == 'train' else val_data
    # Random starting positions
    ix = torch.randint(len(data_split) - block_size, (batch_size,))
    x = torch.stack([data_split[i    : i + block_size    ] for i in ix])
    y = torch.stack([data_split[i + 1: i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

# Show what a batch looks like
xb, yb = get_batch('train', block_size=10, batch_size=2)
print("Input  (x):", xb[0].tolist())
print("Target (y):", yb[0].tolist())
print("\nDecoded x:", repr(decode(xb[0].tolist())))
print("Decoded y:", repr(decode(yb[0].tolist())))
print("\nNote: y is x shifted by 1 — each token predicts the next one")

## Step 4: Define the model

In [ ]:
config = GPTConfig(
    vocab_size  = vocab_size,  # 65 chars
    block_size  = 256,         # context window
    n_layer     = 6,           # transformer blocks
    n_head      = 6,           # attention heads
    n_embd      = 384,         # embedding dimension
    dropout     = 0.2,
)

model = GPT(config).to(device)

print(f"Model configuration:")
print(f"  vocab_size : {config.vocab_size}")
print(f"  block_size : {config.block_size}  (context window)")
print(f"  n_layer    : {config.n_layer}")
print(f"  n_head     : {config.n_head}")
print(f"  n_embd     : {config.n_embd}")
print(f"  dropout    : {config.dropout}")
print(f"")
print(f"Parameters: {model.num_parameters():,} (excl. position embeddings)")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,} (total)")

In [ ]:
# What does an untrained model generate? (random gibberish)
model.eval()
context = torch.zeros((1, 1), dtype=torch.long, device=device)  # start with token 0 ('\n')
with torch.no_grad():
    out = model.generate(context, max_new_tokens=200, temperature=1.0, top_k=40)
print("Untrained model output (random):")
print(decode(out[0].tolist()))

## Step 5: Training

Key training decisions:
- **AdamW optimizer** with weight decay on weights (not biases/norms)
- **Cosine learning rate schedule** with warmup
- **Gradient clipping** at 1.0 to prevent explosions
- Evaluate on validation set periodically

In [ ]:
# Training hyperparameters
MAX_ITERS    = 5000    # total training steps (increase for better results)
BATCH_SIZE   = 32
BLOCK_SIZE   = config.block_size
LEARNING_RATE = 3e-4
WARMUP_ITERS = 100
LR_DECAY_ITERS = MAX_ITERS
MIN_LR       = LEARNING_RATE / 10
EVAL_INTERVAL = 500
EVAL_ITERS   = 50      # batches to average for val loss estimate
GRAD_CLIP    = 1.0

# Separate parameter groups for weight decay
# (biases and LayerNorm params should NOT have weight decay)
decay_params     = [p for n, p in model.named_parameters() if p.dim() >= 2]
no_decay_params  = [p for n, p in model.named_parameters() if p.dim() < 2]

optimizer = torch.optim.AdamW([
    {'params': decay_params,    'weight_decay': 0.1},
    {'params': no_decay_params, 'weight_decay': 0.0},
], lr=LEARNING_RATE, betas=(0.9, 0.95))

def get_lr(it: int) -> float:
    """Cosine learning rate schedule with linear warmup."""
    if it < WARMUP_ITERS:
        return LEARNING_RATE * (it + 1) / WARMUP_ITERS
    if it > LR_DECAY_ITERS:
        return MIN_LR
    decay_ratio = (it - WARMUP_ITERS) / (LR_DECAY_ITERS - WARMUP_ITERS)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return MIN_LR + coeff * (LEARNING_RATE - MIN_LR)

@torch.no_grad()
def estimate_loss() -> dict:
    """Estimate train/val loss over multiple batches."""
    model.eval()
    losses = {}
    for split in ['train', 'val']:
        split_losses = []
        for _ in range(EVAL_ITERS):
            x, y = get_batch(split, BLOCK_SIZE, BATCH_SIZE)
            _, loss = model(x, y)
            split_losses.append(loss.item())
        losses[split] = np.mean(split_losses)
    model.train()
    return losses

print(f"Training setup:")
print(f"  Max iterations:  {MAX_ITERS:,}")
print(f"  Batch size:      {BATCH_SIZE}")
print(f"  Block size:      {BLOCK_SIZE}")
print(f"  Tokens per iter: {BATCH_SIZE * BLOCK_SIZE:,}")
print(f"  LR:              {LEARNING_RATE} (cosine decay with {WARMUP_ITERS} warmup steps)")

In [ ]:
# Training loop
model.train()
best_val_loss = float('inf')
t0 = time.time()

for it in range(MAX_ITERS):
    # Set learning rate
    lr = get_lr(it)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    # Evaluate periodically
    if it % EVAL_INTERVAL == 0:
        losses = estimate_loss()
        elapsed = time.time() - t0
        print(f"step {it:5d} | train loss: {losses['train']:.4f} | "
              f"val loss: {losses['val']:.4f} | lr: {lr:.2e} | "
              f"time: {elapsed:.1f}s")
        
        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            torch.save(model.state_dict(), '../nanogpt/best_model.pt')

    # Forward + backward
    x, y = get_batch('train', BLOCK_SIZE, BATCH_SIZE)
    logits, loss = model(x, y)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    optimizer.step()

print(f"\nTraining complete! Best val loss: {best_val_loss:.4f}")

## Step 6: Generate text from the trained model

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load('../nanogpt/best_model.pt', map_location=device))
model.eval()

def generate_text(prompt: str = '', max_new_tokens: int = 500,
                  temperature: float = 0.8, top_k: int = 40) -> str:
    """Generate text from the trained GPT."""
    if prompt:
        context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    else:
        context = torch.zeros((1, 1), dtype=torch.long, device=device)
    
    with torch.no_grad():
        out = model.generate(
            context,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k
        )
    return decode(out[0].tolist())

print("=" * 60)
print("GENERATED TEXT (no prompt):")
print("=" * 60)
print(generate_text(max_new_tokens=500, temperature=0.8))

In [ ]:
# With a prompt
print("=" * 60)
print("GENERATED TEXT (with prompt 'ROMEO:'):")
print("=" * 60)
print(generate_text(prompt='ROMEO:', max_new_tokens=300, temperature=0.8))

## Step 7: Effect of temperature on generation

In [ ]:
prompt = 'KING HENRY:'
for temp in [0.3, 0.8, 1.2, 1.8]:
    print(f"\n--- temperature={temp} ---")
    text_out = generate_text(prompt=prompt, max_new_tokens=100, temperature=temp)
    print(text_out)
    
print()
print("Low temp → repetitive/predictable")
print("High temp → creative/random/incoherent")

## Step 8: Understanding what the model learned

In [ ]:
# Inspect per-character loss
# A good model should have lower loss on common/predictable sequences

test_strings = [
    'ROMEO: O, she doth teach',   # very Shakespeare-like
    'zzzzzzzzzzzzzzzzzzzzzzzzz',  # uncommon pattern
    '\n\nFirst Citizen:\n',        # common format in the corpus
    'To be or not to be',          # famous phrase
]

model.eval()
print("Loss per test string (lower = more 'expected' by the model):")
for s in test_strings:
    if len(s) < 2:
        continue
    tokens = torch.tensor([encode(s)], dtype=torch.long, device=device)
    x, y = tokens[:, :-1], tokens[:, 1:]
    with torch.no_grad():
        _, loss = model(x, y)
    print(f"  {repr(s):40s} → loss: {loss.item():.3f}")

In [ ]:
# Inspect attention patterns in the final block
# We can hook into the attention module to visualize what tokens attend to what

attention_weights = []

def save_attn_hook(module, input, output):
    # We need to re-compute attention weights for visualization
    pass  # We'll do a simpler inspection below

# Instead, let's look at the learned positional embeddings
pos_emb = model.transformer['wpe'].weight.detach().cpu()  # (block_size, n_embd)

import torch.nn.functional as F
# Cosine similarity between consecutive positions
sims = []
for i in range(min(20, config.block_size - 1)):
    sim = F.cosine_similarity(pos_emb[i].unsqueeze(0), pos_emb[i+1].unsqueeze(0)).item()
    sims.append(sim)

print("Cosine similarity between consecutive positional embeddings:")
print("(nearby positions should be similar — smooth positional structure)")
for i, s in enumerate(sims):
    bar = '█' * int(s * 30)
    print(f"  pos {i:2d} ↔ {i+1:2d}: {s:.3f} {bar}")

## Step 9: Training loss interpretation

In [ ]:
print("Loss interpretation for character-level models:")
print()

baseline_loss = math.log(vocab_size)  # random model
print(f"  Random model loss:   {baseline_loss:.3f}  (log({vocab_size}) = uniform distribution over all chars)")
print(f"  Best val loss:       {best_val_loss:.3f}")
print(f"  Improvement:         {baseline_loss - best_val_loss:.3f} nats")
print()
print(f"  Perplexity (random): {math.exp(baseline_loss):.1f}")
print(f"  Perplexity (model):  {math.exp(best_val_loss):.1f}")
print()
print("Lower perplexity = model is 'less surprised' by the next character.")
print("A random model has perplexity = vocab_size (equally confused about all chars).")

## Key Architecture Summary

```
nanogpt/model.py
├── GPTConfig     — hyperparameter dataclass
├── CausalSelfAttention
│     ├── c_attn: Linear(n_embd, 3*n_embd)  ← Q, K, V in one shot
│     ├── causal mask: lower-triangular matrix
│     └── c_proj: Linear(n_embd, n_embd)
├── MLP
│     ├── c_fc:   Linear(n_embd, 4*n_embd)
│     ├── GELU activation
│     └── c_proj: Linear(4*n_embd, n_embd)
├── Block = LayerNorm → Attention → LayerNorm → MLP (with residuals)
└── GPT
      ├── wte: Embedding(vocab_size, n_embd)   ← token embeddings
      ├── wpe: Embedding(block_size, n_embd)   ← position embeddings
      ├── N × Block
      ├── LayerNorm
      ├── lm_head: Linear(n_embd, vocab_size)  ← tied with wte
      └── generate(): autoregressive sampling
```

**To scale up:** Increase `n_layer`, `n_head`, `n_embd` and `MAX_ITERS`. The full GPT-2 (124M) uses n_layer=12, n_head=12, n_embd=768 on much more data.